# NBA Trade Acquisition Impact Tool — Handoff Guide

**For:** AI integration teammate

**From:** Modeling team

**Purpose:** Everything you need to load the final models and generate predictions and explanations. You do not need to retrain anything.

---

## What the Model Does

The tool takes a player trade and estimates its likely impact on the receiving team's win percentage, based on historical NBA transactions from 2000–2025.

It outputs two things independently:

**Regressor** — predicts the numeric win percentage change for the receiving team after the trade
- Trained on trades only (3,302 rows)
- Model: Gradient Boosting
- MAE: 0.1005 (roughly ±8 wins over an 82-game season)
- R²: 0.180

**Classifier** — predicts the probability that the receiving team improves at all
- Trained on all acquisitions, trades + signings (5,745 rows)
- Model: Random Forest
- AUC: 0.614 | F1: 0.641 | Accuracy: 62.7%

The two models are selected independently and do not need to share the same dataset.

---

## What the Model Does NOT Do

Be careful not to oversell this in the app or report.

- Does **not** guarantee causal trade outcomes
- Does **not** value draft picks or cash considerations
- Does **not** account for injuries, chemistry, or coaching changes
- Uses season-level player profiles, not exact pre-trade snapshots

**Safe framing:** *'Historically, similar acquisitions had an X% probability of improving the receiving team.'*

**Unsafe framing:** *'This trade will succeed.'*

---

## Files You Need


In [ ]:
# These are the only files the app needs.
# Do not retrain — just load these.

ARTIFACTS = {
    'regressor':   'final_trade_impact_regressor.pkl',   # GradBoost, trades only
    'classifier':  'final_trade_impact_classifier.pkl',  # RF, all acquisitions
    'features':    'final_model_features.pkl',           # list of 250 feature names
    'dataset':     'final_modeling_dataset.csv',         # trades-only dataset with all features
}

# The dataset is needed for:
#   - the prediction helper (looks up a player's most recent trade row)
#   - the similarity engine (cosine similarity against historical trades)
# It is NOT used for retraining.

print('Required files:')
for k, v in ARTIFACTS.items():
    print(f'  {k:<14} {v}')


## Setup


In [ ]:
!pip install xgboost scikit-learn pandas numpy --quiet

import numpy as np
import pandas as pd
import joblib
import io
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

print('Setup complete')


## Load Models & Data


In [ ]:
from google.colab import files

print('Upload final_trade_impact_regressor.pkl')
u = files.upload(); reg_pipe = joblib.load(io.BytesIO(list(u.values())[0]))
print('  Loaded regressor:', type(reg_pipe['m']).__name__)


In [ ]:
print('Upload final_trade_impact_classifier.pkl')
u = files.upload(); clf_pipe = joblib.load(io.BytesIO(list(u.values())[0]))
print('  Loaded classifier:', type(clf_pipe['m']).__name__)


In [ ]:
print('Upload final_model_features.pkl')
u = files.upload(); FEATURES = joblib.load(io.BytesIO(list(u.values())[0]))
print(f'  Loaded {len(FEATURES)} features')


In [ ]:
print('Upload final_modeling_dataset.csv')
u = files.upload(); df = pd.read_csv(io.BytesIO(list(u.values())[0]))
print(f'  Loaded dataset: {df.shape}')
print(f'  Seasons: {df["season"].min()}–{df["season"].max()}')
print(f'  Players: {df["player_name"].nunique()} unique')


## Prediction Helper

This is the main function the app should call. Pass a player name and optionally a receiving team or season to narrow the lookup. It finds the most recent matching trade row and runs both models.

The output dict is designed to be passed directly to your AI explanation generator.


In [ ]:
def _label(score):
    if score <= -0.08: return 'strong negative'
    if score <= -0.02: return 'mild negative'
    if score <   0.02: return 'neutral'
    if score <   0.08: return 'mild positive'
    return 'strong positive'

def _impact_score(change, lo=-0.20, hi=0.20):
    """Scale predicted win% change to a 0–100 impact score."""
    return round((max(lo, min(hi, change)) - lo) / (hi - lo) * 100, 1)

def predict_acquisition_impact(player_name, receiving_team=None, season=None):
    """
    Estimate the historical impact of a player acquisition on the receiving team.

    Args:
        player_name:    str  — player to look up (case-insensitive)
        receiving_team: str  — (optional) team name or abbreviation
        season:         int  — (optional) season year

    Returns:
        dict with all fields needed for AI explanation generation.
        Returns {'error': '...'} if the player is not found.
    """
    mask = df['player_name'].str.lower() == player_name.strip().lower()
    if receiving_team:
        mask &= (
            df['secondary_team'].str.lower().str.contains(receiving_team.lower()) |
            df['secondary_abbrev'].str.lower() == receiving_team.lower()
        )
    if season:
        mask &= df['season'] == int(season)

    sub = df[mask].sort_values('season', ascending=False)
    if sub.empty:
        return {'error': f"No trade rows found for '{player_name}' with the given filters."}

    row      = sub.iloc[[0]]
    X_row    = row[FEATURES]
    pred     = float(reg_pipe.predict(X_row)[0])
    prob     = float(clf_pipe.predict_proba(X_row)[0, 1])

    base_pred = row['recv_hist_baseline_win_pct_pred'].values[0]
    trans_wp  = row['receiving_team_trans_win_pct'].values[0]
    pre_wp    = row['receiving_team_pre_win_pct'].values[0] if 'receiving_team_pre_win_pct' in row.columns else None

    return {
        # Identity
        'player':                     row['player_name'].values[0],
        'season':                     int(row['season'].values[0]),
        'from_team':                  row['primary_team'].values[0],
        'to_team':                    row['secondary_team'].values[0],
        'transaction_text':           str(row['text'].values[0]) if 'text' in row.columns else '',

        # Team context at time of trade
        'team_win_pct_prior_season':  round(float(pre_wp), 4)    if pre_wp  is not None and pd.notna(pre_wp)  else None,
        'team_win_pct_trade_season':  round(float(trans_wp), 4)  if pd.notna(trans_wp)  else None,
        'team_baseline_prediction':   round(float(base_pred), 4) if pd.notna(base_pred) else None,
        'team_expected_change':       round(float(base_pred - trans_wp), 4)
                                      if pd.notna(base_pred) and pd.notna(trans_wp) else None,

        # Player context
        'player_bpm':                 round(float(row['player_trans_bpm'].values[0]), 2)
                                      if 'player_trans_bpm' in row.columns else None,
        'player_vorp':                round(float(row['player_trans_vorp'].values[0]), 2)
                                      if 'player_trans_vorp' in row.columns else None,
        'player_pts_per_game':        round(float(row['player_trans_pts_per_game'].values[0]), 1)
                                      if 'player_trans_pts_per_game' in row.columns else None,
        'player_age':                 int(row['player_trans_age'].values[0])
                                      if 'player_trans_age' in row.columns else None,

        # Model outputs
        'predicted_win_pct_change':   round(pred, 4),
        'prob_improvement':           round(prob, 4),
        'impact_score_0_to_100':      _impact_score(pred),
        'interpretation':             _label(pred),

        # Caveats for AI to use
        'model_caveats': [
            'This estimate is based on historical similar acquisitions, not a guaranteed causal outcome.',
            'Draft picks and cash considerations are not valued by the model.',
            'Player stats reflect season-level profiles and may not capture exact pre-trade timing.',
            'The model does not account for injuries, chemistry, or coaching changes.',
        ]
    }

# Example
example = df['player_name'].dropna().iloc[0]
result  = predict_acquisition_impact(example)
print(f'Example output for: {example}\n')
for k, v in result.items():
    if k != 'model_caveats':
        print(f'  {k}: {v}')
print(f'  model_caveats: {len(result["model_caveats"])} caveats included')


## Similarity Helper

Returns the 5 most historically similar trades for any acquisition. Useful for grounding the AI explanation in real examples: *'Here are trades that looked similar and what actually happened.'*


In [ ]:
# Build scaled feature matrix once at load time
_imp  = SimpleImputer(strategy='median')
_sc   = StandardScaler()
_X_sc = _sc.fit_transform(_imp.fit_transform(df[FEATURES]))
print('Similarity matrix built')


In [ ]:
def find_similar_acquisitions(player_name, receiving_team=None, season=None, n=5):
    """
    Return the n most historically similar trade acquisitions.

    Args:
        player_name:    str — player to look up
        receiving_team: str — (optional) team name or abbreviation
        season:         int — (optional) season year
        n:              int — number of comps to return (default 5)

    Returns:
        list of dicts, each describing a similar historical acquisition.
        Returns {'error': '...'} if the player is not found.
    """
    mask = df['player_name'].str.lower() == player_name.strip().lower()
    if receiving_team:
        mask &= (
            df['secondary_team'].str.lower().str.contains(receiving_team.lower()) |
            df['secondary_abbrev'].str.lower() == receiving_team.lower()
        )
    if season:
        mask &= df['season'] == int(season)

    sub = df[mask].sort_values('season', ascending=False)
    if sub.empty:
        return {'error': f"No rows found for '{player_name}'."}

    idx_list  = df.index.tolist()
    query_pos = idx_list.index(sub.index[0])
    sims      = cosine_similarity(_X_sc[query_pos].reshape(1, -1), _X_sc)[0]
    sims[query_pos] = -999

    results = []
    for p in np.argsort(sims)[::-1][:n]:
        row = df.iloc[p]
        results.append({
            'player':                row['player_name'],
            'season':                int(row['season']),
            'from_team':             row['primary_team'],
            'to_team':               row['secondary_team'],
            'actual_win_pct_change': round(float(row['receiving_team_post_win_pct_change']), 4),
            'actual_residual':       round(float(row['residual_win_pct']), 4)
                                     if pd.notna(row.get('residual_win_pct')) else None,
            'similarity_score':      round(float(sims[p]), 4),
            'transaction_text':      str(row.get('text', ''))[:150],
        })
    return results

# Example
comps = find_similar_acquisitions(example)
print(f'Similar acquisitions to {example}:\n')
for c in comps:
    print(f"  {c['player']:<26} s{c['season']}  {c['to_team']:<28}"
          f"  actual={c['actual_win_pct_change']:+.3f}  sim={c['similarity_score']:.3f}")


## How to Use These Outputs for AI Explanation Generation

The `predict_acquisition_impact()` output dict is designed to be passed directly to your AI explanation prompt. Here is a suggested prompt template:


In [ ]:
def build_explanation_prompt(prediction: dict, comps: list) -> str:
    """
    Build a prompt for an AI model to generate a natural-language explanation
    of a trade acquisition prediction.

    Pass the output of predict_acquisition_impact() and find_similar_acquisitions()
    directly into this function.
    """
    if 'error' in prediction:
        return f'Error: {prediction["error"]}'

    p = prediction
    comp_text = '\n'.join([
        f"  - {c['player']} to {c['to_team']} ({c['season']}): "
        f"actual win% change = {c['actual_win_pct_change']:+.3f}, "
        f"similarity = {c['similarity_score']:.3f}"
        for c in comps
    ]) if isinstance(comps, list) else '  None available'

    prompt = f"""You are an NBA trade analyst assistant. Explain the following trade prediction
in clear, concise language for a basketball fan. Be honest about uncertainty.
Do not use bullet points. Write 3–4 sentences.

Trade: {p['player']} from {p['from_team']} to {p['to_team']} (season {p['season']})

Player profile at time of trade:
  Age: {p.get('player_age', 'N/A')}
  Points per game: {p.get('player_pts_per_game', 'N/A')}
  Box Plus/Minus (BPM): {p.get('player_bpm', 'N/A')}
  Value Over Replacement Player (VORP): {p.get('player_vorp', 'N/A')}

Receiving team context:
  Win% prior season: {p.get('team_win_pct_prior_season', 'N/A')}
  Win% in trade season: {p.get('team_win_pct_trade_season', 'N/A')}
  Expected win% next season (before this trade): {p.get('team_baseline_prediction', 'N/A')}
  Expected change before trade: {p.get('team_expected_change', 'N/A')}

Model prediction:
  Predicted win% change: {p['predicted_win_pct_change']:+.4f}
  Probability of improvement: {p['prob_improvement']:.1%}
  Impact score (0–100): {p['impact_score_0_to_100']}
  Interpretation: {p['interpretation']}

Most similar historical acquisitions and their actual outcomes:
{comp_text}

Important caveats to acknowledge:
{chr(10).join('  - ' + c for c in p['model_caveats'])}
"""
    return prompt

# Example — print the prompt that would be sent to the AI
comps  = find_similar_acquisitions(example)
result = predict_acquisition_impact(example)
prompt = build_explanation_prompt(result, comps)
print(prompt)


## Model Performance Reference

Include these numbers in the report. Do not adjust them.


In [ ]:
print('FINAL MODEL PERFORMANCE')
print('=' * 50)
print('Regressor: Gradient Boosting')
print('  Dataset:  Trades only (3,302 rows, 2000–2025)')
print('  Target:   Receiving team win% change')
print('  MAE:      0.1005  (~8 wins over 82 games)')
print('  RMSE:     0.1185')
print('  R2:       0.1801')
print('  Baseline: MAE 0.1018, R2 < 0  (always predict mean)')
print()
print('Classifier: Random Forest')
print('  Dataset:  All acquisitions (5,745 rows, 2000–2025)')
print('  Target:   Did the receiving team improve? (binary)')
print('  AUC:      0.6143')
print('  F1:       0.6414')
print('  Accuracy: 62.7%')
print('  Baseline: 59.7%  (always predict majority class)')
print()
print('Features: 250 (player stats, team trajectory, fit,')
print('          roster activity, SOS/MOV, age context)')
print('Leakage:  None detected')
print('Split:    Time-aware — train pre-2020, test 2020+')
print('=' * 50)


## Caveats for the Report

Copy these directly into your report or app UI.

1. This tool estimates historical player acquisition impact, not a guaranteed causal effect. Many factors beyond the trade affect team win percentage.

2. Draft picks and cash considerations are not directly valued by the model.

3. Player statistics are season-level historical profiles and may not perfectly represent exact pre-trade timing for mid-season transactions.

4. The classifier is trained on both trades and signings to maximise training data. Improvement probabilities should be interpreted as historical base rates for similar acquisitions, not precise predictions.

5. The tool is best used as decision support, not a definitive trade grade. An R² of 0.18 means the model explains a meaningful but partial share of outcome variance — the rest reflects injuries, chemistry, schedule, and other factors no model can predict without information that does not exist at trade time.
